In [1]:
import pandas as pd
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from datasets import Dataset
from transformers import BertTokenizer, TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")

In [4]:
print(df.columns)
print(df.shape)

Index(['verse_id', 'song_id', 'ori_track_name', 'clean_track_name',
       'all_artists', 'primary_artist', 'artist_genres', 'main_genre',
       'explicit', 'section', 'verse', 'language', 'language.1', 'confidence',
       'confidence.1', 'label'],
      dtype='str')
(22878, 16)


In [5]:
# Class balance check
print(df['label'].value_counts())

label
0    11439
1    11439
Name: count, dtype: int64


In [6]:
# Select only the columns we need
df = df[['verse', 'label']]

In [7]:
# Convert to Hugging Face format
dataset = Dataset.from_pandas(df)

In [8]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

In [9]:
# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [10]:
# Initiate tokenizer with the cache path
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', cache_dir=cache_dir)

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\Documents\devanasokan_fyp\huggingface_cache\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [11]:
def preprocess_function(examples):
    # This creates the 'input_ids' and 'attention_mask' BERT needs
    return tokenizer(examples["verse"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 22878/22878 [00:06<00:00, 3653.75 examples/s]


In [12]:
# 80% Train, 20% Test
full_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

In [13]:
print(full_dataset) 
# If it shows {'train': ..., 'test': ...}, it is already split!

DatasetDict({
    train: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 18302
    })
    test: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4576
    })
})


In [14]:
model_name = "distilbert-base-uncased" # Or any model from the Hugging Face Hub

# 1. Load the tokenizer (must match the model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1546.36it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [16]:
import evaluate
metric = evaluate.load("accuracy")

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # argmax picks the highest probability (0 or 1)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Folder where checkpoints are saved
    eval_strategy="epoch",           # Run evaluation after every epoch
    save_strategy="epoch",           # Save model after every epoch
    learning_rate=2e-5,               # Default value; overridden in the final run
    per_device_train_batch_size=16,   # Default value; overridden in the final run
    per_device_eval_batch_size=16,    # Batch size for evaluation
    num_train_epochs=2,               # Default value; overridden in the final run
    weight_decay=0.01,                # Regularization to prevent overfitting
    load_best_model_at_end=True,      # Keeps the best version of the model
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()    # Use Mixed Precision if on GPU for 2x speed
)


In [19]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

print(trainer.compute_metrics)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5137.18it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<function compute_metrics at 0x00000188A4814AE0>


In [20]:
# Hyperparameter search space definition
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [8, 16]
        ),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }

# Run the hyperparameter search
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=6,
    compute_objective=lambda metrics: metrics["eval_accuracy"],
)

print("Best run: ", best_run)
print("Best hyperparameters found: ", best_run.hyperparameters)

[I 2026-07-18 21:42:52,919] A new study created in memory with name: no-name-97f62416-e567-48e0-9a37-ed421abe874d
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5398.49it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.333736,0.305674,0.865822
2,0.227890,0.320692,0.873689


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-07-18 21:59:22,651] Trial 0 finished with value: 0.8736888111888111 and parameters: {'learning_rate': 1.255355081205268e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 2, 'weight_decay': 0.042221587664312867}. Best is trial 0 with value: 0.8736888111888111.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1887.07it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  |

Epoch,Training Loss,Validation Loss,Accuracy
1,0.336250,0.309535,0.863636
2,0.234243,0.319606,0.876748
3,0.188304,0.352127,0.877622


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-07-18 22:31:12,810] Trial 1 finished with value: 0.8776223776223776 and parameters: {'learning_rate': 1.0712430568886262e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.06660092909461611}. Best is trial 1 with value: 0.8776223776223776.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1302.25it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  |

...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.341136,0.331946,0.851617
2,0.185564,0.374130,0.877404
3,0.108147,0.475547,0.876748
4,0.044536,0.662702,0.880682


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.04it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-07-18 23:58:01,725] Trial 4 finished with value: 0.8806818181818182 and parameters: {'learning_rate': 4.43170998444413e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.016900358498568968}. Best is trial 2 with value: 0.8813374125874126.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1786.70it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | 

Epoch,Training Loss,Validation Loss,Accuracy
1,0.336866,0.319039,0.857299


[I 2026-07-19 00:04:34,536] Trial 5 pruned. 


Best run:  BestRun(run_id='2', objective=0.8813374125874126, hyperparameters={'learning_rate': 1.5125430047068985e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.07744747806156313}, run_summary=None)
Best hyperparameters found:  {'learning_rate': 1.5125430047068985e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.07744747806156313}


In [21]:
best_hyperparameters = best_run.hyperparameters
final_training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_hyperparameters["learning_rate"],
    per_device_train_batch_size=best_hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    num_train_epochs=best_hyperparameters["num_train_epochs"],
    weight_decay=best_hyperparameters["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

final_trainer = Trainer(
    model=model_init(),
    args=final_training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

final_trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1456.41it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.331573,0.302088,0.867788
2,0.212709,0.326733,0.879371
3,0.162298,0.396977,0.882649


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3432, training_loss=0.24596193175771575, metrics={'train_runtime': 2218.3985, 'train_samples_per_second': 24.75, 'train_steps_per_second': 1.547, 'total_flos': 7273254990606336.0, 'train_loss': 0.24596193175771575, 'epoch': 3.0})

In [22]:
history = pd.DataFrame(final_trainer.state.log_history)
print(history)


       loss  grad_norm  learning_rate     epoch  step  eval_loss  \
0  0.390947   8.511459       0.000013  0.437063   500        NaN   
1  0.331573   7.217982       0.000011  0.874126  1000        NaN   
2       NaN        NaN            NaN  1.000000  1144   0.302088   
3  0.261531   8.955596       0.000009  1.311189  1500        NaN   
4  0.212709  10.035900       0.000006  1.748252  2000        NaN   
5       NaN        NaN            NaN  2.000000  2288   0.326733   
6  0.192129   4.305753       0.000004  2.185315  2500        NaN   
7  0.162298  17.577181       0.000002  2.622378  3000        NaN   
8       NaN        NaN            NaN  3.000000  3432   0.396977   
9       NaN        NaN            NaN  3.000000  3432        NaN   

   eval_accuracy  eval_runtime  eval_samples_per_second  \
0            NaN           NaN                      NaN   
1            NaN           NaN                      NaN   
2       0.867788       41.6599                  109.842   
3            Na

In [23]:
# Confusion matrix and classification report
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
classification_report_output = classification_report(full_dataset["test"]["label"], np.argmax(final_trainer.predict(full_dataset["test"]).predictions, axis=-1))
print(classification_report_output)

print("Confusion Matrix:")
confusion_matrix_output = confusion_matrix(full_dataset["test"]["label"], np.argmax(final_trainer.predict(full_dataset["test"]).predictions, axis=-1))
print(confusion_matrix_output)

Classification Report:


              precision    recall  f1-score   support

           0       0.88      0.89      0.88      2268
           1       0.89      0.88      0.88      2308

    accuracy                           0.88      4576
   macro avg       0.88      0.88      0.88      4576
weighted avg       0.88      0.88      0.88      4576

Confusion Matrix:


[[2011  257]
 [ 280 2028]]


In [23]:
# Save the version currently in the final trainer's brain
final_trainer.save_model("./my_final_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]


In [24]:
# TEST MODEL

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from your local folder
path = "./my_final_model"
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Ensure you saved the tokenizer there too!

# Create a 'pipeline' (the easiest way to use the model)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7364.39it/s]


In [25]:
# Test it on a new sentence
result = classifier("kiss it from my lips")
print(result)

[{'label': 'LABEL_1', 'score': 0.9721267223358154}]


In [34]:
import accelerate
print(accelerate.__version__)

1.13.0


In [35]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.3.0


In [36]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
